# Transcricao PT-BR -> legenda .srt (faster-whisper large-v3)

**Fluxo:** o video `.mp4` vira `.mp3` **no seu PC** (`mp4_para_mp3.py`).
Aqui no Colab entra **so o .mp3** - nenhum frame de video sobe para a nuvem.

**Para rodar:** menu `Ambiente de execucao` > `Executar tudo`.
Este notebook ja pede a **GPU T4** sozinho. Se mesmo assim a celula 3 avisar que falta GPU:
`Ambiente de execucao` > `Alterar o tipo de ambiente de execucao` > **T4 GPU** > `Salvar`.

Configuracao fixa deste notebook:
- modelo `large-v3`, idioma travado em portugues (`language="pt"`);
- VAD ligado (corta silencio, reduz alucinacao do modelo);
- timestamps por palavra -> legendas curtas (max. 2 linhas de 40 caracteres);
- **a legenda 1 comeca em `00:00:00,000`** - a nao ser que o video abra com vinheta ou musica
  longa, e ai a legenda 1 espera a fala comecar de verdade;
- o `.srt` baixa sozinho no fim.

> Arquivo grande (mais de ~100 MB)? Arraste o `.mp3` para o painel **Arquivos** (icone de pasta
> na barra esquerda) antes de rodar a celula 2 - ela usa o mp3 que ja estiver em `/content`.

> Se aparecer o aviso **"voce esta conectado a uma GPU mas nao esta usando"**, feche e ignore.
> Aceitar a troca reinicia o ambiente e apaga o mp3 que voce subiu.


In [ ]:
#@title 1. Ambiente: GPU + dependencias { display-mode: "form" }
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "SEM GPU -> Ambiente de execucao > Alterar o tipo > T4 GPU"

# ctranslate2 >= 4.6.3 e o que evita o crash de cuDNN no Colab (esses wheels sao compilados
# com WITH_CUDNN=OFF). O ffmpeg nao entra aqui: o faster-whisper decodifica com PyAV.
!pip -q install "faster-whisper==1.2.1" "ctranslate2>=4.6.3"

import ctranslate2, faster_whisper, torch
print("faster-whisper:", faster_whisper.__version__,
      "| ctranslate2:", ctranslate2.__version__,
      "| CUDA:", torch.cuda.is_available())


In [ ]:
#@title 2. Enviar o .mp3 gerado no seu PC { display-mode: "form" }
from pathlib import Path
from google.colab import files

# Ja tem um .mp3 em /content (painel Arquivos ou Drive)? usa ele. Senao, abre o seletor.
existentes = sorted(Path("/content").glob("*.mp3"))
if len(existentes) > 1:
    print("varios mp3 em /content:", ", ".join(p.name for p in existentes))
    print("-> vou usar o primeiro. Apague os outros no painel Arquivos se nao for esse.\n")
if existentes:
    ENTRADA = existentes[0]
    print("usando o mp3 que ja esta no Colab:", ENTRADA.name)
else:
    ENTRADA = Path(next(iter(files.upload())))

print(f"arquivo: {ENTRADA.name}  ({ENTRADA.stat().st_size/1e6:.1f} MB)")

In [ ]:
#@title 3. Transcrever com large-v3 travado em portugues { display-mode: "form" }
import time

import torch
from faster_whisper import WhisperModel

if not torch.cuda.is_available():
    raise SystemExit(
        "SEM GPU. Menu Ambiente de execucao > Alterar o tipo de ambiente de execucao > T4 GPU,"
        " depois Ambiente de execucao > Executar tudo."
    )

# O .mp3 vai direto para o modelo: o faster-whisper decodifica com PyAV e ja reamostra para
# 16 kHz mono por dentro - converter para WAV antes seria so uma etapa a mais para dar errado.
#
# min_silence_duration_ms=2000 e o padrao da lib: com os 500 antigos, o Silero fundia dois
# trechos sempre que o intervalo entre eles era menor que 2*speech_pad_ms (800 ms), e a pausa
# sumia - justamente as pausas de 0,5 a 1,5 s que o GAP_QUEBRA da celula 4 usa para quebrar.
# float16 e o formato nativo da T4: melhor qualidade, e os 3 GB do modelo cabem folgado
# nos 16 GB da placa - quantizar para int8 aqui nao compra nada.
modelo = WhisperModel("large-v3", device="cuda", compute_type="float16")

gerador, info = modelo.transcribe(
    str(ENTRADA),
    language="pt",                 # travado: nao tenta detectar outro idioma
    task="transcribe",
    beam_size=5,
    word_timestamps=True,          # necessario para as legendas curtas
    vad_filter=True,               # corta silencio -> menos alucinacao
    vad_parameters=dict(
        threshold=0.5,                  # o quanto o VAD precisa ter certeza de que aquilo e fala
        min_silence_duration_ms=2000,   # so corta o audio em silencios de 2 s+ (ver comentario acima)
        speech_pad_ms=400,              # nao reduza: com 200 ms o modelo cortava a primeira palavra
    ),
    condition_on_previous_text=False,   # evita o modelo repetir frases em loop
    # a escada precisa terminar em 1.0: se a janela ainda falha no ultimo degrau, o
    # faster-whisper ACEITA o melhor dos fracassos e o loop de repeticao entra no .srt.
    temperature=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
)
print(f"duracao do audio: {info.duration/60:.1f} min - transcrevendo...\n")

t0 = time.time()
segmentos = []
for seg in gerador:                # o gerador so processa quando iterado
    segmentos.append(seg)
    pct = 100 * seg.end / info.duration
    print(f"[{pct:5.1f}%] [{seg.start:7.1f}s] {seg.text.strip()[:80]}")
print(f"\n{len(segmentos)} segmentos em {time.time()-t0:.0f}s")


In [ ]:
#@title 4. Montar o .srt (legendas curtas, ancoradas em 00:00:00,000) { display-mode: "form" }
# Regras de exibicao da legenda (padrao proximo ao de streaming)
MAX_CHARS = 40          # caracteres por linha (max_line_width)
MAX_LINHAS = 2          # linhas por legenda (max_line_count)
MAX_DUR = 6.0           # segundos por legenda
MIN_DUR = 0.7           # nenhuma legenda pisca mais rapido que isso
GAP_QUEBRA = 0.7        # silencio entre palavras que forca nova legenda
ESPERA_ANCORA = 6.0     # so puxa a legenda 1 para o zero se a fala comecar ate aqui
FIM_FRASE = ".?!:;"     # pontuacao que fecha a legenda


def coletar_palavras(segmentos):
    """Achata os segmentos em uma lista unica de palavras com tempo."""
    palavras = []
    for seg in segmentos:
        if seg.words:
            for w in seg.words:
                texto = w.word.strip()
                if texto:
                    palavras.append({"t": texto, "ini": w.start, "fim": w.end})
        else:  # segmento sem timestamp por palavra: entra inteiro
            texto = seg.text.strip()
            if texto:
                palavras.append({"t": texto, "ini": seg.start, "fim": seg.end})
    return palavras


def _fatiar(palavra):
    """Palavra maior que a linha inteira (URL, nome tecnico): corta em pedacos."""
    if len(palavra) <= MAX_CHARS:
        return [palavra]
    return [palavra[i:i + MAX_CHARS] for i in range(0, len(palavra), MAX_CHARS)]


def _envolver(texto):
    """Quebra gulosa: linhas de no maximo MAX_CHARS caracteres."""
    linhas, atual = [], ""
    for bruta in texto.split():
        for palavra in _fatiar(bruta):
            if not atual:
                atual = palavra
            elif len(atual) + 1 + len(palavra) <= MAX_CHARS:
                atual += " " + palavra
            else:
                linhas.append(atual)
                atual = palavra
    if atual:
        linhas.append(atual)
    return linhas


def cabe(texto):
    """O texto cabe na tela dentro do limite de linhas?"""
    return len(_envolver(texto)) <= MAX_LINHAS


def quebrar_linhas(texto):
    """Divide em linhas equilibradas (visual melhor que a quebra gulosa)."""
    if len(texto) <= MAX_CHARS:
        return texto
    palavras = texto.split()
    melhor, dif_melhor = None, None
    for corte in range(1, len(palavras)):
        a, b = " ".join(palavras[:corte]), " ".join(palavras[corte:])
        if len(a) > MAX_CHARS or len(b) > MAX_CHARS:
            continue
        dif = abs(len(a) - len(b))
        if dif_melhor is None or dif < dif_melhor:
            melhor, dif_melhor = (a, b), dif
    if melhor:
        return melhor[0] + "\n" + melhor[1]
    return "\n".join(_envolver(texto))


def agrupar(palavras):
    """Junta palavras em blocos de legenda respeitando tela, duracao e pausas."""
    blocos, atual = [], []

    for p in palavras:
        if atual:
            candidato = " ".join(x["t"] for x in atual) + " " + p["t"]
            estourou = (
                not cabe(candidato)
                or p["fim"] - atual[0]["ini"] > MAX_DUR
                or p["ini"] - atual[-1]["fim"] > GAP_QUEBRA
            )
            if estourou:
                blocos.append(atual)
                atual = []
        atual.append(p)
        texto_atual = " ".join(x["t"] for x in atual)
        if p["t"][-1] in FIM_FRASE and len(texto_atual) > MAX_CHARS:
            blocos.append(atual)
            atual = []

    if atual:
        blocos.append(atual)
    return blocos


def montar_legendas(blocos, ancorar_no_zero=True):
    """Blocos -> lista de legendas com tempos limpos, sem sobreposicao."""
    legendas = []
    for bloco in blocos:
        texto = " ".join(p["t"] for p in bloco)
        legendas.append({"ini": bloco[0]["ini"], "fim": bloco[-1]["fim"], "txt": texto})

    if not legendas:
        return legendas

    # A legenda 1 comeca em 00:00:00,000 - a menos que o video abra com vinheta ou
    # musica longa: esticar a legenda por 20 s dava spoiler e ficava fora de sincronia.
    if ancorar_no_zero and legendas[0]["ini"] <= ESPERA_ANCORA:
        legendas[0]["ini"] = 0.0

    anterior_fim = 0.0
    for leg in legendas:
        leg["ini"] = round(max(leg["ini"], anterior_fim), 3)
        fim = max(leg["fim"], leg["ini"] + MIN_DUR)
        leg["fim"] = round(min(fim, leg["ini"] + MAX_DUR), 3)   # nada fica na tela alem do limite
        anterior_fim = leg["fim"]
    return legendas


def tempo_srt(t):
    ms = int(round(t * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def escrever_srt(legendas, caminho):
    linhas = []
    for i, leg in enumerate(legendas, 1):
        linhas.append(str(i))
        linhas.append(f"{tempo_srt(leg['ini'])} --> {tempo_srt(leg['fim'])}")
        linhas.append(quebrar_linhas(leg["txt"]))
        linhas.append("")
    with open(caminho, "w", encoding="utf-8") as f:
        f.write("\n".join(linhas))
    return caminho

palavras = coletar_palavras(segmentos)
legendas = montar_legendas(agrupar(palavras), ancorar_no_zero=True)
SRT = escrever_srt(legendas, ENTRADA.with_suffix(".srt"))

inicio_da_fala = min(p["ini"] for p in palavras)
print(f"{len(palavras)} palavras -> {len(legendas)} legendas")
print("primeira legenda comeca em:", tempo_srt(legendas[0]["ini"]))
if inicio_da_fala <= ESPERA_ANCORA:
    assert tempo_srt(legendas[0]["ini"]) == "00:00:00,000", "ancoragem no zero falhou"
else:
    print(f"(a fala so comeca em {tempo_srt(inicio_da_fala)} - vinheta/musica no inicio,"
          " entao a legenda 1 nao foi puxada para o zero)")
assert all(leg["fim"] - leg["ini"] <= MAX_DUR + 1e-6 for leg in legendas), "legenda longa demais"


In [ ]:
#@title 5. Conferir e baixar o arquivo .srt { display-mode: "form" }
from google.colab import files

print(SRT.read_text(encoding="utf-8")[:600], "...\n")
print("Se o download nao comecar sozinho: abra o painel Arquivos (icone de pasta, a esquerda),"
      f" clique com o botao direito em {SRT.name} e escolha Fazer download.")
files.download(str(SRT))
